# Лекција 01 - Увод у AI агенте

Добродошли у прву лекцију курса **AI агенти за почетнике**!

**AI агент** је програм који користи велики језички модел (LLM) као свој мотор размишљања и може предузимати *акције* у стварном свету — позивати API-је, претраживати базе података или покретати код — како би остварио циљ у име корисника.

У овом записнику ћете креирати свог првог агента: **Туристичког агента** који препоручује дестинације за одмор. Током процеса ћете научити како да:

1. Повежете се са Microsoft Foundry Agent Service користећи **Microsoft Agent Framework**.
2. Дате агенту **алат** — обичну Python функцију коју може позвати.
3. Покренете агента и прегледате његов одговор.
4. Стримујете одговор агента токен по токен.


## Подешавање

Пре покретања овог бележника, уверите се да имате:

1. **Microsoft Foundry пројекат** са распоређеним моделом за ћаскање (нпр. `gpt-4.1-mini`).
2. **Пријаву преко Azure CLI** — покрените `az login` у вашем терминалу.
3. **Постављене потребне променљиве окружења:**
   - `AZURE_AI_PROJECT_ENDPOINT` — крајња тачка вашег Microsoft Foundry пројекта.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — име вашег распоређеног модела.

Испод ћелије инсталира Python пакете који су вам потребни.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import dotenv
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from agent_framework import tool

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME in your .env file."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential()
)

## Креирање Вашег Првог Агента

Агенту су потребне две ствари:

- **Упутства** која му говоре *ко је* и *како да се понаша* (системски подсетник).
- **Алатке** — Python функције украшене са `@tool` које агент може позвати да добије информације или изврши акције.

Испод дефинишемо једну једноставну алатку која враћа листу популарних дестинација за одмор. Агент ће користити ову алатку када корисник затражи препоруке за путовања.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = provider.as_agent(
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
    tools=[get_destinations],
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## Одговори у стриму

За интерактивније искуство можете **стримовати** одговор агента. Уместо да чекате цео одговор, агент испоручује делове текста како се генеришу. Ово је посебно корисно у интерфејсима за ћаскање где желите да приказујете излаз у реалном времену.


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## Резиме

У овој лекцији сте научили како да:

- **Креирате провајдера** који се повезује са Microsoft Foundry Agent Service преко `FoundryChatClient`.
- **Дефинишете алат** користећи `@tool` декоратор тако да агент може позивати ваше Python функције.
- **Покренете агента** са поруком корисника и одштампате његов одговор.
- **Стримујете одговоре** за излаз у реалном времену.

У следећој лекцији ћемо детаљније истражити агентске оквире и научити како агенти могу добити моћније алате и способности за мулти-степено расуђивање.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Изјава о одрицању одговорности**:
Овај документ је преведен коришћењем услуге за аутоматски превод [Co-op Translator](https://github.com/Azure/co-op-translator). Иако тежимо тачности, имајте у виду да аутоматски преводи могу садржати грешке или нетачности. Оригинални документ на његовом изворном језику треба сматрати ауторитативним извором. За критичне информације препоручује се професионални људски превод. Нисмо одговорни за било каква неспоразума или погрешна тумачења која произилазе из коришћења овог превода.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
